In [3]:
import numpy as np
import pandas as pd

# Zadanie 1
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)
display(titanic.groupby(['Pclass', 'Sex'])['Age'].mean())

Pclass  Sex   
1       female    34.611765
        male      41.281386
2       female    28.722973
        male      30.740707
3       female    21.750000
        male      26.507589
Name: Age, dtype: float64

In [5]:
# Zadanie 2
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)

# tabela przestawna (grupuję według klasy, rozdzielam pasażerów według przeżycia i zliczam w każdej kombinacji)
survival_pivot = titanic.pivot_table(
    index='Pclass',
    columns='Survived',
    aggfunc='size') 

print(survival_pivot)

Survived    0    1
Pclass            
1          80  136
2          97   87
3         372  119


In [7]:
# Zadanie 3
np.random.seed(42)
sales = pd.DataFrame({'Region': np.random.choice(['A', 'B', 'C'], 100), 'Sprzedaż': np.random.randint(50, 500, 100)})
round(sales.groupby('Region')['Sprzedaż'].agg(['min', 'max', 'mean', 'std']), 2)

,min,max,mean,std
Region,,,,
A,64,495,285.94,128.35
B,54,495,255.89,137.73
C,77,498,262.35,126.50


In [8]:
# Zadanie 4
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)

# obliczam średnią dla każdej klasy i przypisuję ją do kazdego wiersza
titanic['Avg_Fare_Pclass'] = titanic.groupby('Pclass')['Fare'].transform('mean')
titanic[['Pclass', 'Fare', 'Avg_Fare_Pclass']].head(10)

,Pclass,Fare,Avg_Fare_Pclass
0,3,7.2500,13.675550
1,1,71.2833,84.154687
2,3,7.9250,13.675550
3,1,53.1000,84.154687
4,3,8.0500,13.675550
5,3,8.4583,13.675550
6,1,51.8625,84.154687
7,3,21.0750,13.675550
8,3,11.1333,13.675550
9,2,30.0708,20.662183


In [12]:
# Zadanie 5
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)
titanic_fe = titanic.copy()

# Metoda pd.cut() z własnymi przedziałami
titanic_fe['Age_Group'] = pd.cut(
    titanic_fe['Age'],
    bins=[0, 12, 18, 60, 100],
    labels=['Dziecko', 'Nastolatek', 'Dorosły', 'Senior']
)

# Analiza survival rate według grup wiekowych
survival_age = titanic_fe.groupby('Age_Group', observed=False)['Survived'].agg(['count', 'mean'])
survival_age.columns = ['Liczba', 'Survival_Rate']
print("\n=== Survival rate według grup wiekowych ===")
display(survival_age.round(3))

survival_age['Survival_Rate'] *= 100
for i in range(4):
    bar = int((survival_age['Survival_Rate'].iloc[i]) // 2)
    print(f" {survival_age.index[i]:<10}  {'█' * bar}")


=== Survival rate według grup wiekowych ===


,Liczba,Survival_Rate
Age_Group,,
Dziecko,69,0.580
Nastolatek,70,0.429
Dorosły,553,0.389
Senior,22,0.227


 Dziecko     ████████████████████████████
 Nastolatek  █████████████████████
 Dorosły     ███████████████████
 Senior      ███████████


In [14]:
# Zadanie 6
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)

# tworzę nową kolumnę - liczba osób w rodzinie
titanic_family = titanic.copy()
titanic_family['Family_Size'] = titanic_family['SibSp'] + titanic_family['Parch'] + 1

# każdemu pasażerowi przypisuję survival_rate dla jego rozmiaru rodziny
titanic_family['Survival_Rate'] = titanic_family.groupby('Family_Size')['Survived'].transform('mean')

# identyfikuję , który rozmiar rodziny miał najwyższy survival rate
df_survival_rate = titanic_family[['Family_Size', 'Survival_Rate']].value_counts().reset_index()
df_survival_rate = df_survival_rate.sort_values(['Family_Size'])
display(df_survival_rate)

max_survival_rate_idx = df_survival_rate['Survival_Rate'].idxmax() # zwraca wiersz z max survival_rate

print(f"Najwyższy survival_rate miał rozmiar rodziny: {int(df_survival_rate.loc[max_survival_rate_idx]['Family_Size'])}")

# wizualizacja
df_survival_rate['Survival_Rate'] *= 100
for i in range(len(df_survival_rate)):
    bar = int((df_survival_rate['Survival_Rate'].iloc[i]) // 2)
    print(f" {df_survival_rate['Family_Size'][i]:<10}  {'█' * bar}")

,Family_Size,Survival_Rate,count
0,1,0.303538,537
1,2,0.552795,161
2,3,0.578431,102
3,4,0.724138,29
5,5,0.200000,15
4,6,0.136364,22
6,7,0.333333,12
8,8,0.000000,6
7,11,0.000000,7


Najwyższy survival_rate miał rozmiar rodziny: 4
 1           ███████████████
 2           ███████████████████████████
 3           ████████████████████████████
 4           ████████████████████████████████████
 6           ██████████
 5           ██████
 7           ████████████████
 11          
 8           


In [16]:
# Zadanie 7
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)




In [18]:
# === GENEROWANIE SYNTETYCZNEGO DATASETU CALIFORNIA HOUSING ===

def make_california_housing(n=5000, seed=42):
    """Generuje syntetyczny dataset wzorowany na California Housing."""
    
    rng = np.random.default_rng(seed) # tworzy generator liczb losowych NumPy
    # Współrzędne geograficzne w obszarze Kalifornii (szerokość ~32-42, długość ~-124..-114)
    latitude = rng.uniform(32.5, 41.5, n) # uniform() – losuje wartości z rozkładu jednostajnego
    longitude = rng.uniform(-123.5, -114.5, n)
    
    # Dochód: rozkład skośny
    med_inc = rng.gamma(shape=2.0, scale=1.8, size=n) + 0.5
    
    # Wiek domu
    house_age = rng.integers(1, 55, n)
    
    # Pokoje/sypialnie/populacja
    ave_rooms = rng.normal(5.5, 1.5, n).clip(1, 15)
    ave_bedrms = (ave_rooms * rng.uniform(0.15, 0.25, n)).clip(0.5, 5)
    population = rng.integers(50, 5000, n)
    ave_occup = rng.normal(3.0, 0.8, n).clip(1, 10)
    
    # Cena domu (w 100_000 USD) - zależna od dochodu, wieku i lokalizacji
    med_house_val = (
        med_inc * 0.6
        + (latitude - 36) * 0.15
        - (house_age / 100)
        + rng.normal(0, 0.3, n)
    ).clip(0.15, 5.0)
    
    return pd.DataFrame({
        'MedInc': med_inc.round(4),
        'HouseAge': house_age,
        'AveRooms': ave_rooms.round(4),
        'AveBedrms': ave_bedrms.round(4),
        'Population': population,
        'AveOccup': ave_occup.round(4),
        'Latitude': latitude.round(4),
        'Longitude': longitude.round(4),
        'MedHouseVal': med_house_val.round(4)
    })

df_housing = make_california_housing(n=5000, seed=42)
print("=== Syntetyczny California Housing Dataset ===")

# tworzymy kategorie geograficzne
df_housing['lat_bin'] = pd.cut(df_housing['Latitude'], 4)
df_housing['lon_bin'] = pd.cut(df_housing['Longitude'], 4)

#df_housing.loc[(df_housing['Latitude'] >= 36) & (df_housing['Longitude'] >= -119), 'Region'] = 'NE'
#df_housing.loc[(df_housing['Latitude'] >= 36) & (df_housing['Longitude'] < -119), 'Region'] = 'NW'
#df_housing.loc[(df_housing['Latitude'] < 36) & (df_housing['Longitude'] >= -119), 'Region'] = 'SE'
#df_housing.loc[(df_housing['Latitude'] < 36) & (df_housing['Longitude'] < -119), 'Region'] = 'SW'
df_housing['Region'] = df_housing['lat_bin'].astype(str) + '|' + df_housing['lon_bin'].astype(str)

df_housing['Avg_MedHouseVal'] = df_housing.groupby('Region')['MedHouseVal'].transform('mean')
display(df_housing.sort_values('Avg_MedHouseVal', ascending=False).head())

=== Syntetyczny California Housing Dataset ===


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,lat_bin,lon_bin,Region,Avg_MedHouseVal
542,4.1052,34,7.8794,1.5543,4221,2.5246,40.0516,-118.1141,2.3854,"(39.25, 41.498]","(-118.999, -116.749]","(39.25, 41.498]|(-118.999, -116.749]",2.757198
3027,3.7330,44,4.5010,0.9161,2992,4.0970,40.2371,-116.9342,2.5076,"(39.25, 41.498]","(-118.999, -116.749]","(39.25, 41.498]|(-118.999, -116.749]",2.757198
3005,3.3641,18,3.8461,0.6235,1030,2.4957,39.4698,-117.1786,2.0476,"(39.25, 41.498]","(-118.999, -116.749]","(39.25, 41.498]|(-118.999, -116.749]",2.757198
3009,2.5086,40,7.9927,1.9312,4497,2.8010,39.2575,-118.2089,1.3759,"(39.25, 41.498]","(-118.999, -116.749]","(39.25, 41.498]|(-118.999, -116.749]",2.757198
4508,2.3915,28,4.8913,0.8651,4585,2.3065,39.3068,-116.9925,1.9529,"(39.25, 41.498]","(-118.999, -116.749]","(39.25, 41.498]|(-118.999, -116.749]",2.757198


In [20]:
# Zadanie 8
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)
titanic_missing = titanic.copy()

# wypełnianie braków grupowo
print(f"Liczba braków w Age: {len(titanic) -  titanic['Age'].count()}")
titanic_missing['Age'] = titanic_missing.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
print(f"Liczba braków w Age po wypełnieniu: {len(titanic_missing) - titanic_missing['Age'].count()}")

# describe
display(titanic.describe())
display(titanic_missing.describe())

Liczba braków w Age: 177
Liczba braków w Age po wypełnieniu: 0


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.066409,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,13.244532,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,22.000000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,26.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,37.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [22]:
# Zadanie 9
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)
titanic_multi = titanic.copy()

# tworze kolumnę IsAlone
titanic_multi['IsAlone'] = titanic_multi['SibSp'] + titanic_multi['Parch'] + 1
titanic_multi.head(3)

titanic_multi_group = titanic_multi.groupby(['Pclass', 'Sex', 'IsAlone']).agg({'Age': ['mean'], 'Survived': ['sum', 'count']})


titanic_multi_group = titanic_multi_group.reset_index()
titanic_multi_group['SurvivalRate'] = titanic_multi_group['Survived']['sum'] / titanic_multi_group['Survived']['count']
display(titanic_multi_group)

Pclass     Sex IsAlone        Age Survived       SurvivalRate
                                mean      sum count             
0       1  female       1  34.937500       33    34     0.970588
1       1  female       2  36.906250       39    39     1.000000
2       1  female       3  36.923077       13    13     1.000000
3       1  female       4  19.250000        2     4     0.500000
4       1  female       5  19.500000        2     2     1.000000
5       1  female       6  23.500000        2     2     1.000000
6       1    male       1  44.601852       25    75     0.333333
7       1    male       2  37.483871       12    31     0.387097
8       1    male       3  42.545455        5    11     0.454545
9       1    male       4  15.973333        3     3     1.000000
10      1    male       6  41.500000        0     2     0.000000
11      2  female       1  33.383333       29    32     0.906250
12      2  female       2  28.105263       17    19     0.894737
13      2  female       3  22.785714       13    14     0.928571
14      2  female       4  21.444444        9     9     1.000000
15      2  female       5  54.000000        1     1     1.000000
16      2  female       6  24.000000        1     1     1.000000
17      2    male       1  33.904762        7    72     0.097222
18      2    male       2  33.700000        1    15     0.066667
19      2    male       3  18.637059        8    17     0.470588
20      2    male       4  21.250000        1     4     0.250000
21      3  female       1  23.565789       37    60     0.616667
22      3  female       2  22.214286       15    29     0.517241
23      3  female       3  20.764706       12    22     0.545455
24      3  female       4  12.700000        5     6     0.833333
25      3  female       5  19.666667        0     9     0.000000
26      3  female       6  27.200000        0     5     0.000000
27      3  female       7  15.875000        3     8     0.375000
28      3  female       8  29.500000        0     2     0.000000
29      3  female      11        NaN        0     3     0.000000
30      3    male       1  29.184492       32   264     0.121212
31      3    male       2  24.496190        5    28     0.178571
32      3    male       3  23.071429        8    25     0.320000
33      3    male       4  13.500000        1     3     0.333333
34      3    male       5   9.000000        0     3     0.000000
35      3    male       6   9.583333        0    12     0.000000
36      3    male       7  13.750000        1     4     0.250000
37      3    male       8   8.750000        0     4     0.000000
38      3    male      11        NaN        0     4     0.000000

In [24]:
# Zadanie 10

# === GENEROWANIE SYNTETYCZNEGO DATASETU CALIFORNIA HOUSING ===

def make_california_housing(n=5000, seed=42):
    """Generuje syntetyczny dataset wzorowany na California Housing."""
    
    rng = np.random.default_rng(seed) # tworzy generator liczb losowych NumPy
    # Współrzędne geograficzne w obszarze Kalifornii (szerokość ~32-42, długość ~-124..-114)
    latitude = rng.uniform(32.5, 41.5, n) # uniform() – losuje wartości z rozkładu jednostajnego
    longitude = rng.uniform(-123.5, -114.5, n)
    
    # Dochód: rozkład skośny
    med_inc = rng.gamma(shape=2.0, scale=1.8, size=n) + 0.5
    
    # Wiek domu
    house_age = rng.integers(1, 55, n)
    
    # Pokoje/sypialnie/populacja
    ave_rooms = rng.normal(5.5, 1.5, n).clip(1, 15)
    ave_bedrms = (ave_rooms * rng.uniform(0.15, 0.25, n)).clip(0.5, 5)
    population = rng.integers(50, 5000, n)
    ave_occup = rng.normal(3.0, 0.8, n).clip(1, 10)
    
    # Cena domu (w 100_000 USD) - zależna od dochodu, wieku i lokalizacji
    med_house_val = (
        med_inc * 0.6
        + (latitude - 36) * 0.15
        - (house_age / 100)
        + rng.normal(0, 0.3, n)
    ).clip(0.15, 5.0)
    
    return pd.DataFrame({
        'MedInc': med_inc.round(4),
        'HouseAge': house_age,
        'AveRooms': ave_rooms.round(4),
        'AveBedrms': ave_bedrms.round(4),
        'Population': population,
        'AveOccup': ave_occup.round(4),
        'Latitude': latitude.round(4),
        'Longitude': longitude.round(4),
        'MedHouseVal': med_house_val.round(4)
    })

df_housing = make_california_housing(n=5000, seed=42)
df_housing

# tworzenie regionów
df_housing['Region'] = 'Unknown'
df_housing.loc[(df_housing['Latitude'] >= 36) & (df_housing['Longitude'] >= -119), 'Region'] = 'NE'
df_housing.loc[(df_housing['Latitude'] >= 36) & (df_housing['Longitude'] < -119), 'Region'] = 'NW'
df_housing.loc[(df_housing['Latitude'] < 36) & (df_housing['Longitude'] >= -119), 'Region'] = 'SE'
df_housing.loc[(df_housing['Latitude'] < 36) & (df_housing['Longitude'] < -119), 'Region'] = 'SW'

display(df_housing)

# obliczmay kwartyle Q1 i Q3 oraz IQR w funkcji iqr_func
def iqr_func(x):
    Q1 = x.quantile(0.25)    
    Q3 = x.quantile(0.75)    
    IQR = Q3 - Q1 
    return IQR

df_housing.groupby('Region')['MedHouseVal'].agg(lambda x: iqr_func(x))


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,Region
0,0.9067,48,4.1561,0.9944,3929,2.5674,39.4656,-121.5367,0.4077,NW
1,7.1731,51,6.9524,1.3357,3825,4.9413,36.4499,-117.5337,4.0278,NE
2,4.8817,22,5.0764,0.9349,3550,3.2407,40.2274,-116.5982,3.2236,NE
3,4.0980,54,3.3280,0.8064,3875,2.5277,38.7763,-122.0003,3.0991,NW
4,5.1915,5,6.3899,1.1873,4516,3.7003,33.3476,-123.1053,2.2712,SW
...,...,...,...,...,...,...,...,...,...,...
4995,5.3653,13,4.7637,1.1841,243,1.9708,34.5119,-119.4793,2.9516,SW
4996,5.5702,7,7.9692,1.2887,1125,2.7444,34.0155,-120.7159,2.8750,SW
4997,3.6182,2,7.5327,1.2491,2592,3.3957,36.7403,-122.0119,2.1636,NW
4998,1.6971,30,5.8728,1.0326,2635,3.0316,35.6787,-120.1393,1.1051,SW


Region
NE    2.029500
NW    1.922925
SE    1.934700
SW    1.862050
Name: MedHouseVal, dtype: float64

In [26]:
# Zadanie 11 - korzystamy z danych z poprzedniego zadania

def min_max(x):
    minimum = min(x)
    maksimum = max(x)

    wartosc = (x - minimum) / (maksimum - minimum)
    return wartosc


df_housing["MedInc_norm"] = df_housing.groupby('Region')['MedInc'].transform(lambda x: min_max(x))
display(df_housing)

# sprawdzam min i max dla każdego regionu oddzielnie
df_housing.groupby('Region')['MedInc_norm'].agg(['min', 'max'])

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,Region,MedInc_norm
0,0.9067,48,4.1561,0.9944,3929,2.5674,39.4656,-121.5367,0.4077,NW,0.011401
1,7.1731,51,6.9524,1.3357,3825,4.9413,36.4499,-117.5337,4.0278,NE,0.378662
2,4.8817,22,5.0764,0.9349,3550,3.2407,40.2274,-116.5982,3.2236,NE,0.248023
3,4.0980,54,3.3280,0.8064,3875,2.5277,38.7763,-122.0003,3.0991,NW,0.131479
4,5.1915,5,6.3899,1.1873,4516,3.7003,33.3476,-123.1053,2.2712,SW,0.280772
...,...,...,...,...,...,...,...,...,...,...,...
4995,5.3653,13,4.7637,1.1841,243,1.9708,34.5119,-119.4793,2.9516,SW,0.291391
4996,5.5702,7,7.9692,1.2887,1125,2.7444,34.0155,-120.7159,2.8750,SW,0.303911
4997,3.6182,2,7.5327,1.2491,2592,3.3957,36.7403,-122.0119,2.1636,NW,0.113426
4998,1.6971,30,5.8728,1.0326,2635,3.0316,35.6787,-120.1393,1.1051,SW,0.067260


,min,max
Region,,
NE,0.0,1.0
NW,0.0,1.0
SE,0.0,1.0
SW,0.0,1.0


In [28]:
# Zadaniue 12

def iqr_func(x):
    Q1 = x.quantile(0.25)    
    Q3 = x.quantile(0.75)    
    IQR = Q3 - Q1 
    przedzial_min = Q1 - 1.5 * IQR
    przedzial_max = Q3 + 1.5 * IQR

    outliery = []
    for i in x:
        if (i < przedzial_min) or (i > przedzial_max):
            outliery.append(i)
        
    return outliery, len(outliery)

wynik = df_housing.groupby('Region')['MedHouseVal'].agg(lambda x: iqr_func(x))

liczby = []
for x in wynik:
    liczby.append(x[1])
pd.DataFrame({'Liczba outlierów': liczby}, index=wynik.index)  



print("=== Rozkład sprzedaży według regionu (tekstowy boxplot) ===\n")
stats = df_housing.groupby('Region')['MedHouseVal'].agg([
    'min',
    lambda x: x.quantile(0.25),
    'median',
    lambda x: x.quantile(0.75),
    'max'
])

stats.columns = ['min', 'Q1', 'med', 'Q3', 'max']
print(stats.round(1))

# Rysujemy prostą tekstową skalę
overall_min = df_housing['MedHouseVal'].min()
overall_max = df_housing['MedHouseVal'].max()
width = 50 # szerokość tekstowej skali

def pos(val):
    """Zwraca pozycję (0..width-1) dla danej wartości na skali"""
    return int((val - overall_min) / (overall_max - overall_min) * (width - 1))

print("\nTekstowy boxplot (| = kwartyle, - = zakres, * = mediana):")
for region, row in stats.iterrows():
    line = [' '] * width
    # zakres min..max = "-"
    for i in range(pos(row['min']), pos(row['max']) + 1):
        line[i] = '-'
    # Q1..Q3 = "="
    for i in range(pos(row['Q1']), pos(row['Q3']) + 1):
        line[i] = '='
    # mediana = "*"
    line[pos(row['med'])] = '*'
    # krawędzie
    line[pos(row['min'])] = '|'
    line[pos(row['max'])] = '|'
    print(f" {region:10s} |{''.join(line)}|")

=== Rozkład sprzedaży według regionu (tekstowy boxplot) ===

        min   Q1  med   Q3  max
Region                         
NE      0.2  1.4  2.3  3.4  5.0
NW      0.2  1.4  2.2  3.3  5.0
SE      0.2  0.8  1.7  2.7  5.0
SW      0.2  0.9  1.6  2.7  5.0

Tekstowy boxplot (| = kwartyle, - = zakres, * = mediana):
 NE         ||-----------=========*============---------------||
 NW         ||-----------========*============----------------||
 SE         ||-----=========*===========----------------------||
 SW         ||------=======*===========-----------------------||


In [32]:
# Zadanie 13
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv( url)

titanic['Family_Size'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['Is_Alone'] = titanic['Family_Size'] == 1
titanic['Title'] = titanic['Name'].str.split(',').str[1].str.split(' ').str[1].str.strip() # wyciągam np Mr.
titanic['Fare_Per_Person'] = titanic['Fare'] / titanic['Family_Size']
titanic['Age_Group'] = pd.cut(
    titanic['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Dziecko', 'Nastolatek', 'Młody dorosły', 'Dorosły', 'Senior'])
titanic['Survival_Rate_By_Title'] = titanic.groupby('Title')['Survived'].transform('mean')
titanic.head(3)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Is_Alone,Title,Fare_Per_Person,Age_Group,Survival_Rate_By_Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,Mr.,3.62500,Młody dorosły,0.156673
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,False,Mrs.,35.64165,Dorosły,0.792000
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,True,Miss.,7.92500,Młody dorosły,0.697802
